# HealthBench worst-30% open-Fusion challenge

Can a fusion of open-weights models improve on the baseline across the 157 hardest
[HealthBench](https://openai.com/index/healthbench/) Professional conversations? 

## 0. Before running

From a terminal in `packages/screamingface/`:

```bash
just stack-prepare  # first run only: download pinned Benchmark assets
just stack-up       # start AI Gateway :9105 and Engine :9108
just stack-status
```

Use `just stack-logs` to inspect startup failures and `just stack-down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

In [ ]:
import screamingface as sf

sf.connect()

In [ ]:
from helpers import openrouter_credits

openrouter_credits(api_key="your key here")

## 1. Run the benchmark with 1 model

In [ ]:
PARAMS = {"max_tokens": 16384, "temperature": 0.0}

deepseek = sf.Model(
    model="openrouter/deepseek/deepseek-v4-pro",
    params=PARAMS,
)

In [ ]:
report = sf.evaluate(deepseek, benchmark="healthbench-worst30", limit=1)
report

## 2. Define the Fusion with open source models and evaluate it

In [ ]:
qwen = sf.Model(
    model="openrouter/qwen/qwen3.8-2.4t-a95b",
    params=PARAMS,
)
glm = sf.Model(
    model="openrouter/z-ai/glm-5.2",
    params=PARAMS,
)

In [ ]:
SYNTHESIS_PROMPT = (
    "You are synthesising the single best response to a health-related "
    "conversation by combining independent responses from a panel of models. "
    "A physician-authored rubric will grade your output per criterion — "
    "accuracy, completeness, safety, and appropriate hedging all matter; so "
    "does avoiding harmful or false content.\n\n"
    "Procedure:\n"
    "1. Read every panel response carefully.\n"
    "2. Identify what each contributes that the others miss (facts, safety "
    "caveats, follow-up questions, care-seeking guidance).\n"
    "3. Produce ONE unified response to the final user turn that:\n"
    "   - Keeps every clinically correct, relevant point\n"
    "   - Drops anything inaccurate, unsafe, or speculative\n"
    "   - Resolves disagreements by favouring the safer, better-supported claim\n"
    "   - Stays concise — verbosity is not rewarded\n"
    "4. Do not introduce claims no panel member made.\n\n"
    "Output: the unified response only, no preamble, no JSON wrapper."
)

kimi = sf.Model(
    model="openrouter/moonshotai/kimi-k3",
    params=PARAMS,
    prompt=SYNTHESIS_PROMPT,
)

best_open_source = sf.Fusion(
    members=[deepseek, qwen, glm], name="best_open_source", synthesizer=kimi
)

In [ ]:
report = sf.evaluate(best_open_source, benchmark="healthbench-worst30", limit=1)
report